# Step 6: Fraud Ops AI/BI dashboard (Layer 5) reading live from Lakebase

Validation and live-read proof for the dashboard **FE Bar Fraud Ops: Triage & Action** (dashboard ID `01f1ce7f835a1e81b05ebb664bf0994c`, warehouse `9bbc027402111984`).

* Part A: pre-flight checks (stop if any fail)
* Part B: `fraud_dash` read layer in Lakebase (`lakebase_dashboard.sql`): apply, row counts, deny tests, TXN-0234209, latency through UC
* Part D: every dashboard dataset through the warehouse Statement Execution API, live-read proof (agent writes a case, no sync/job in between), TXN-0234209 end-to-end trace

Workspace fevm-workshop-guy, serverless compute. All timestamps are UTC (Postgres session TZ = GMT, warehouse `current_date()` = UTC).

In [0]:
# Same libraries as fraud_triage_agent.ipynb (agent for Part D.2) plus psycopg for direct Postgres checks
%pip install -q databricks-langchain "langgraph>=0.3,<0.4" databricks-agents "psycopg[binary]>=3.1.0" pydantic databricks-ai-search
dbutils.library.restartPython()

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
import os, sys, json, time, statistics, datetime as dt
sys.path.insert(0, os.path.dirname(os.getcwd()))   # repo root (one level above validation_notebooks/)
REPO = os.path.dirname(os.getcwd())

import mlflow
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.sql import StatementParameterListItem
from lakebase_cases import LakebaseCases

w = WorkspaceClient()
WAREHOUSE_ID = "9bbc027402111984"
DASH_ID = "01f1ce7f835a1e81b05ebb664bf0994c"
UC = "fe_bar_lakebase"
MLFLOW_RUN = "d89beaca8f81447780da60ed2ea7c811"
JOB_ID = 1115396029932218
SYNC_PIPELINE = "f8690359-4bb1-45b9-8fdb-5bd892239db2"
TRACED_TXN = "TXN-0234209"
LOG_DATE = "2026-10-23"  # evidence re-run; keeps the committed 2026-10-22 dataset log unchanged
FRAUD_DASH_VIEWS = ["case_summary", "case_actions_daily", "score_distribution", "exposure", "baseline_comparison", "triage_queue"]

def now():
    return dt.datetime.now(dt.timezone.utc).strftime("%Y-%m-%d %H:%M:%S.%f")[:-3] + " UTC"

def fmt_table(cols, rows, max_w=60):
    """Plain-text table (markdown pipe style) so outputs are readable in the committed .ipynb."""
    def s(v):
        v = "NULL" if v is None else str(v)
        return v if len(v) <= max_w else v[: max_w - 3] + "..."
    rows = [[s(v) for v in r] for r in rows]
    out = ["| " + " | ".join(cols) + " |", "| " + " | ".join("---" for _ in cols) + " |"]
    out += ["| " + " | ".join(r) + " |" for r in rows]
    return "\n".join(out)

def show(cols, rows, title=None, max_w=60):
    if title: print(f"\n### {title}")
    print(fmt_table(cols, rows, max_w))

# --- Postgres (direct, identity-aware via SET ROLE) ---------------------------
admin = LakebaseCases()                       # alex.owen login, no SET ROLE
lead = LakebaseCases(pg_role="fraud_lead")    # SET ROLE fraud_lead

def pg(conn_obj, sql, params=()):
    rows = conn_obj.query(sql, params)
    return (list(rows[0].keys()) if rows else []), [list(r.values()) for r in rows]

# --- UC on serverless (Spark) -------------------------------------------------
def uc_rows(sql, args=None):
    df = spark.sql(sql, args=args) if args else spark.sql(sql)
    rows = df.collect()
    return df.columns, [list(r) for r in rows]

# --- Warehouse Statement Execution API (same engine as the dashboard) ----------
def wh(sql, params=None):
    t0 = time.time()
    r = w.statement_execution.execute_statement(warehouse_id=WAREHOUSE_ID, statement=sql, wait_timeout="50s", parameters=params)
    while r.status.state.value in ("PENDING", "RUNNING"):
        time.sleep(1); r = w.statement_execution.get_statement(r.statement_id)
    ms = (time.time() - t0) * 1000
    if r.status.state.value != "SUCCEEDED":
        raise RuntimeError(f"{r.status.state.value}: {r.status.error.message if r.status.error else ''}")
    cols = [c.name for c in r.manifest.schema.columns]
    return cols, (r.result.data_array or []) if r.result else [], ms, r.statement_id

# --- Dashboard dataset SQL, read from the dashboard definition itself ----------
dash_def = json.loads(w.lakeview.get(DASH_ID).serialized_dashboard)
DATASETS = {d["name"]: {"display": d["displayName"], "sql": "".join(d["queryLines"]).strip(), "params": d.get("parameters", [])} for d in dash_def["datasets"]}

def bind_multi(sql, action=(), queue=(), status=()):
    """MULTI (array) parameters of ds_triage, bound the way the dashboard binds them (empty array = no filter).
    Only the parameter markers are replaced; the rest of the SQL is the dashboard's text."""
    arr = lambda xs: "array(" + ", ".join("'" + x + "'" for x in xs) + ")" if xs else "array()"
    return sql.replace(":action_f", arr(action)).replace(":queue_f", arr(queue)).replace(":status_f", arr(status))

def dataset_call(name, txn_id="ALL", **multi):
    """Return (sql, statement-API params, spark args) for a dashboard dataset."""
    sql = DATASETS[name]["sql"]
    if name == "ds_triage":
        return bind_multi(sql, **multi), [StatementParameterListItem(name="txn_id_f", value=txn_id, type="STRING")], {"txn_id_f": txn_id}
    return sql, None, None

print(f"[{now()}] setup done. Dashboard '{w.lakeview.get(DASH_ID).display_name}' has {len(DATASETS)} datasets: {list(DATASETS)}")

[2026-10-23 23:30:43.986 UTC] setup done. Dashboard 'FE Bar Fraud Ops: Triage & Action' has 10 datasets: ['ds_kpi_cases', 'ds_model_vs_baseline', 'ds_score_dist', 'ds_cases_daily', 'ds_actions_daily', 'ds_triage', 'ds_triage_options', 'ds_recent_cases', 'ds_calibration', 'ds_exposure']


In [0]:
print(f"[{now()}] Part A pre-flight")
cols, rows = uc_rows("SELECT current_user() AS current_user, is_account_group_member('pii') AS in_pii")
show(cols, rows, "A.1 current_user / pii membership (UC, serverless)")

counts = []
for t in ["fraud_gov.txn_risk_current", "fraud.txn_risk_current", "cases.fraud_cases"]:
    n = uc_rows(f"SELECT count(*) AS n FROM {UC}.{t}")[1][0][0]
    counts.append([f"{UC}.{t}", n])
show(["object (through UC)", "rows"], counts, "A.2 row counts through UC")
UC_CASES_BEFORE = counts[2][1]
assert UC_CASES_BEFORE == 0, "STOP: fe_bar_lakebase.cases.fraud_cases is no longer 0 rows through UC"
print(f"\nPASS: cases through UC = {UC_CASES_BEFORE} (RLS, reader role has no policy). This is the 'before' picture for Part B.")

[2026-10-23 23:30:47.579 UTC] Part A pre-flight

### A.1 current_user / pii membership (UC, serverless)
| current_user | in_pii |
| --- | --- |
| alex.owen@databricks.com | True |

### A.2 row counts through UC
| object (through UC) | rows |
| --- | --- |
| fe_bar_lakebase.fraud_gov.txn_risk_current | 463881 |
| fe_bar_lakebase.fraud.txn_risk_current | 464632 |
| fe_bar_lakebase.cases.fraud_cases | 0 |

PASS: cases through UC = 0 (RLS, reader role has no policy). This is the 'before' picture for Part B.


In [0]:
cols, rows = pg(lead, "SELECT current_user, session_user")
show(cols, rows, "A.3 Postgres identity (SET ROLE fraud_lead)")
cols, rows = pg(lead, "SELECT assigned_queue, status, count(*) AS cases FROM cases.fraud_cases GROUP BY 1, 2 ORDER BY 1, 2")
show(cols, rows, "A.3 cases by assigned_queue and status (fraud_lead)")
LEAD_TOTAL_BEFORE = sum(r[2] for r in rows)
cols, rows = pg(lead, "SELECT current_date AS today_utc, count(*) FILTER (WHERE created_at::date = current_date) AS created_today, count(*) AS total FROM cases.fraud_cases")
show(cols, rows, "A.3 cases created today (fraud_lead)")
cols, rows = pg(lead, "SELECT case_id, txn_id, status, assigned_queue, recommended_action, created_by, created_at, db_inserted_by FROM cases.fraud_cases WHERE txn_id = %s", (TRACED_TXN,))
show(cols, rows, f"A.3 case row for {TRACED_TXN} (fraud_lead)")
assert rows and str(rows[0][0]).startswith("2319d4b7"), "STOP: traced case 2319d4b7 not found"


### A.3 Postgres identity (SET ROLE fraud_lead)
| current_user | session_user |
| --- | --- |
| fraud_lead | alex.owen@databricks.com |

### A.3 cases by assigned_queue and status (fraud_lead)
| assigned_queue | status | cases |
| --- | --- | --- |
| analyst_review | closed | 213 |
| auto_block | closed | 5 |
| auto_block | open | 8 |

### A.3 cases created today (fraud_lead)
| today_utc | created_today | total |
| --- | --- | --- |
| 2026-10-23 | 57 | 226 |

### A.3 case row for TXN-0234209 (fraud_lead)
| case_id | txn_id | status | assigned_queue | recommended_action | created_by | created_at | db_inserted_by |
| --- | --- | --- | --- | --- | --- | --- | --- |
| 2319d4b7-bea3-412b-ba6b-8325e1a7a71d | TXN-0234209 | open | auto_block | block | fraud_triage_copilot:agent | 2026-10-21 00:02:21.031588+00:00 | alex.owen@databricks.com |


In [0]:
import threading
cols, rows = pg(admin, "SELECT oid, datname FROM pg_database WHERE datname = current_database()")
READER = f"databricks_reader_{rows[0][0]}"
print(f"fe_bar_db oid = {rows[0][0]}  ->  UC reader role = {READER}")
cols, rows = pg(admin, """SELECT r.rolname, r.rolsuper, r.rolbypassrls, r.rolcanlogin, r.rolinherit,
                                 coalesce((SELECT string_agg(m.rolname, ', ') FROM pg_auth_members am JOIN pg_roles m ON m.oid = am.roleid WHERE am.member = r.oid), '(none)') AS member_of
                          FROM pg_roles r WHERE r.rolname = %s""", (READER,))
show(cols, rows, "A.4 reader role attributes")
assert rows and rows[0][1] is False and rows[0][2] is False and rows[0][5] == "(none)", "STOP: reader role has elevated attributes or memberships"

# Proof: run a UC query in a background thread and sample pg_stat_activity while it executes.
seen, res = set(), []
th = threading.Thread(target=lambda: res.append(uc_rows(f"SELECT count(*) AS n, sum(amount) AS s FROM {UC}.fraud.txn_features")[1]))
th.start()
while th.is_alive():
    for r in admin.query("SELECT usename, application_name, left(query, 90) AS q FROM pg_stat_activity WHERE datname = 'fe_bar_db' AND pid <> pg_backend_pid() AND usename <> session_user AND query ILIKE '%%txn_features%%'"):
        seen.add((r["usename"], r["application_name"], r["q"].strip()))
    time.sleep(0.2)
th.join()
show(["usename (Postgres session serving the UC query)", "application_name", "query pushed down by UC"], sorted(seen), "A.4 pg_stat_activity sampled during a UC query")
assert any(s[0] == READER for s in seen), "could not observe the UC session"
print(f"\nCONFIRMED: UC queries on {UC} run in Postgres as {READER} (observed in pg_stat_activity while the UC query ran, "
      f"from the JDBC driver, with the pushed-down SQL). UC result: {res[0]}")

fe_bar_db oid = 17063  ->  UC reader role = databricks_reader_17063

### A.4 reader role attributes
| rolname | rolsuper | rolbypassrls | rolcanlogin | rolinherit | member_of |
| --- | --- | --- | --- | --- | --- |
| databricks_reader_17063 | False | False | True | True | (none) |

### A.4 pg_stat_activity sampled during a UC query
| usename (Postgres session serving the UC query) | application_name | query pushed down by UC |
| --- | --- | --- |
| databricks_reader_17063 | PostgreSQL JDBC Driver | SELECT COUNT(*),SUM("amount") FROM "fraud"."txn_features" |

CONFIRMED: UC queries on fe_bar_lakebase run in Postgres as databricks_reader_17063 (observed in pg_stat_activity while the UC query ran, from the JDBC driver, with the pushed-down SQL). UC result: [[464632, 101444581.35999957]]


In [0]:
cols, rows, ms, sid = wh(f"SELECT count(*) AS n, current_user() AS run_as FROM {UC}.fraud_gov.txn_risk_current")
show(cols, rows, f"A.5 warehouse {WAREHOUSE_ID} SELECT on {UC}.fraud_gov.txn_risk_current ({ms:.0f} ms, statement {sid})")
assert int(rows[0][0]) > 0, "STOP: warehouse returned no rows"
print("PASS: all Part A pre-flight checks passed")


### A.5 warehouse 9bbc027402111984 SELECT on fe_bar_lakebase.fraud_gov.txn_risk_current (7105 ms, statement 01f1cf39-d9e2-192a-b993-33b992c285dc)
| n | run_as |
| --- | --- |
| 463881 | alex.owen@databricks.com |
PASS: all Part A pre-flight checks passed


In [0]:
print(f"[{now()}] Part B")
policies_before = pg(admin, "SELECT tablename, policyname, cmd, roles::text, qual, with_check FROM pg_policies WHERE schemaname = 'cases' ORDER BY 1, 2")[1]

# Idempotent apply (CREATE IF NOT EXISTS / CREATE OR REPLACE / re-runnable GRANTs)
dash_sql = open(os.path.join(REPO, "lakebase_dashboard.sql")).read()
conn = admin._get_conn()
with conn.cursor() as cur:
    cur.execute(dash_sql)
conn.commit()
print(f"[{now()}] applied lakebase_dashboard.sql ({len(dash_sql)} chars)")

# lakebase_fgac.sql must carry an identical copy so a fresh rebuild includes fraud_dash
fgac = open(os.path.join(REPO, "lakebase_fgac.sql")).read()
block = fgac.split("-- >>> BEGIN lakebase_dashboard.sql\n")[1].split("-- <<< END lakebase_dashboard.sql\n")[0]
END_MARK = "-- ============================================================\n-- END\n-- ============================================================\n"
assert block == dash_sql.replace(END_MARK, "").rstrip() + "\n", "lakebase_fgac.sql B.4 block differs from lakebase_dashboard.sql"
print("PASS: lakebase_fgac.sql B.4 block is identical to lakebase_dashboard.sql")

# Thresholds: read from the MLflow run, never hardcoded
p = mlflow.get_run(MLFLOW_RUN).data.params
BLOCK_T, REVIEW_T = float(p["block_threshold"]), float(p["review_threshold"])
with conn.cursor() as cur:
    cur.execute("""INSERT INTO fraud_dash.model_thresholds (model_run_id, block_threshold, review_threshold)
                   VALUES (%s, %s, %s)
                   ON CONFLICT (model_run_id) DO UPDATE SET block_threshold = EXCLUDED.block_threshold,
                       review_threshold = EXCLUDED.review_threshold, loaded_at = now()""", (MLFLOW_RUN, BLOCK_T, REVIEW_T))
conn.commit()
show(*pg(admin, "SELECT model_run_id, block_threshold, review_threshold, loaded_at FROM fraud_dash.model_thresholds"), f"fraud_dash.model_thresholds (from MLflow run {MLFLOW_RUN})")

cols, rows = pg(admin, """SELECT c.relname AS object, CASE c.relkind WHEN 'v' THEN 'view' WHEN 'r' THEN 'table' END AS kind,
                                 pg_get_userbyid(c.relowner) AS owner, coalesce(array_to_string(c.reloptions, ','), '') AS options,
                                 has_table_privilege(%s, c.oid, 'SELECT') AS reader_select
                          FROM pg_class c JOIN pg_namespace n ON n.oid = c.relnamespace
                          WHERE n.nspname = 'fraud_dash' AND c.relkind IN ('v', 'r') ORDER BY 1""", (READER,))
show(cols, rows, "fraud_dash objects")
policies_after = pg(admin, "SELECT tablename, policyname, cmd, roles::text, qual, with_check FROM pg_policies WHERE schemaname = 'cases' ORDER BY 1, 2")[1]
assert policies_before == policies_after, "RLS policies changed!"
print(f"PASS: the {len(policies_after)} existing RLS policies on cases.* are unchanged (no new policy added)")

[2026-10-23 23:31:42.393 UTC] Part B
[2026-10-23 23:31:42.724 UTC] applied lakebase_dashboard.sql (12395 chars)
PASS: lakebase_fgac.sql B.4 block is identical to lakebase_dashboard.sql

### fraud_dash.model_thresholds (from MLflow run d89beaca8f81447780da60ed2ea7c811)
| model_run_id | block_threshold | review_threshold | loaded_at |
| --- | --- | --- | --- |
| d89beaca8f81447780da60ed2ea7c811 | 0.9502674433961342 | 0.8714158290727556 | 2026-10-23 23:31:43.235216+00:00 |

### fraud_dash objects
| object | kind | owner | options | reader_select |
| --- | --- | --- | --- | --- |
| baseline_comparison | view | fraud_lead | security_barrier=true | True |
| case_actions_daily | view | fraud_lead | security_barrier=true | True |
| case_summary | view | fraud_lead | security_barrier=true | True |
| exposure | view | fraud_lead | security_barrier=true | True |
| model_thresholds | table | alex.owen@databricks.com |  | True |
| score_distribution | view | fraud_lead | security_barrier=true | Tru

In [0]:
rows = []
for v in FRAUD_DASH_VIEWS + ["model_thresholds"]:
    rows.append([f"{UC}.fraud_dash.{v}", uc_rows(f"SELECT count(*) FROM {UC}.fraud_dash.{v}")[1][0][0]])
show(["object (through UC, as " + READER + ")", "rows"], rows, "B.1 fraud_dash row counts through UC")

DASH_CASES = uc_rows(f"SELECT count(*) FROM {UC}.fraud_dash.case_summary")[1][0][0]
LEAD_CASES = pg(lead, "SELECT count(*) FROM cases.fraud_cases")[1][0][0]
cols, hi = pg(lead, """SELECT c.case_id, c.txn_id, c.status, c.assigned_queue, tf.amount
                       FROM cases.fraud_cases c JOIN fraud_gov.txn_features tf ON tf.txn_id = c.txn_id
                       WHERE tf.amount > 10000 ORDER BY tf.amount DESC""")
show(["source", "cases"], [["fraud_dash.case_summary through UC", DASH_CASES], ["cases.fraud_cases in Postgres as fraud_lead", LEAD_CASES], ["difference", LEAD_CASES - DASH_CASES]], "B.1 case count comparison")
show(cols, hi, "High-amount cases (amount > 10000) visible to fraud_lead, excluded from fraud_dash")
assert LEAD_CASES - DASH_CASES == len(hi), "difference is not fully explained by high-amount exclusions"
print(f"\nPASS: the difference of {LEAD_CASES - DASH_CASES} is exactly the {len(hi)} high-amount cases (amount > 10000 via fraud_gov.txn_features).")


### B.1 fraud_dash row counts through UC
| object (through UC, as databricks_reader_17063) | rows |
| --- | --- |
| fe_bar_lakebase.fraud_dash.case_summary | 220 |
| fe_bar_lakebase.fraud_dash.case_actions_daily | 8 |
| fe_bar_lakebase.fraud_dash.score_distribution | 31 |
| fe_bar_lakebase.fraud_dash.exposure | 6 |
| fe_bar_lakebase.fraud_dash.baseline_comparison | 2 |
| fe_bar_lakebase.fraud_dash.triage_queue | 24201 |
| fe_bar_lakebase.fraud_dash.model_thresholds | 1 |

### B.1 case count comparison
| source | cases |
| --- | --- |
| fraud_dash.case_summary through UC | 220 |
| cases.fraud_cases in Postgres as fraud_lead | 226 |
| difference | 6 |

### High-amount cases (amount > 10000) visible to fraud_lead, excluded from fraud_dash
| case_id | txn_id | status | assigned_queue | amount |
| --- | --- | --- | --- | --- |
| 0a4ac09a-07db-4bd2-a7be-0a3212083825 | TXN-0256919 | closed | analyst_review | 10903.33 |
| 018b3ed8-fae2-43ba-99d8-e50afea74fff | TXN-0142590 | closed | auto_bloc

In [0]:
results = []
def expect(name, fn, ok_if):
    try:
        out = fn(); verdict = "PASS" if ok_if(out, None) else "FAIL"; detail = str(out)[:160]
    except Exception as e:
        verdict = "PASS" if ok_if(None, e) else "FAIL"; detail = (type(e).__name__ + ": " + str(e).split("\n")[0])[:160]
    results.append([name, verdict, detail])

zero_rows = lambda out, e: e is None and out[1] in ([], [[0]])
errored = lambda out, e: e is not None

# 1. raw case table through UC still returns nothing (RLS: reader has no policy)
expect("UC: SELECT count(*) FROM cases.fraud_cases", lambda: uc_rows(f"SELECT count(*) FROM {UC}.cases.fraud_cases"), zero_rows)
expect("UC: SELECT notes, db_inserted_by FROM cases.fraud_cases", lambda: uc_rows(f"SELECT notes, db_inserted_by FROM {UC}.cases.fraud_cases"), zero_rows)
expect("UC: SELECT notes, db_inserted_by FROM cases.case_actions", lambda: uc_rows(f"SELECT db_inserted_by FROM {UC}.cases.case_actions"), zero_rows)
# 2. the dashboard layer does not expose notes / created_by / db_inserted_by
for col in ["notes", "created_by", "db_inserted_by"]:
    expect(f"UC: SELECT {col} FROM fraud_dash.case_summary", lambda col=col: uc_rows(f"SELECT {col} FROM {UC}.fraud_dash.case_summary"), errored)
# 3. writes through UC are rejected (statements are no-ops even if they were allowed)
expect("UC: INSERT INTO fraud_dash.model_thresholds", lambda: uc_rows(f"INSERT INTO {UC}.fraud_dash.model_thresholds VALUES ('deny_test', 0, 0, TIMESTAMP'2000-01-01')"), errored)
expect("UC: UPDATE cases.fraud_cases SET notes = notes", lambda: uc_rows(f"UPDATE {UC}.cases.fraud_cases SET notes = notes WHERE txn_id = '{TRACED_TXN}'"), errored)
expect("UC: DELETE FROM fraud_dash.case_summary WHERE false", lambda: uc_rows(f"DELETE FROM {UC}.fraud_dash.case_summary WHERE 1 = 0"), errored)
show(["deny test", "result", "detail"], results, "B.2 deny tests through UC", max_w=160)
# clean-up guard: if the INSERT had somehow succeeded, remove the row and fail loudly
if pg(admin, "SELECT count(*) FROM fraud_dash.model_thresholds WHERE model_run_id = 'deny_test'")[1][0][0]:
    raise AssertionError("UC INSERT succeeded - investigate")

# 4. Postgres catalog: the reader role has no write privilege anywhere and no column access via fraud_dash
cols, rows = pg(admin, """SELECT n.nspname AS schema, count(*) AS relations,
         count(*) FILTER (WHERE has_table_privilege(%s, c.oid, 'SELECT')) AS reader_select,
         count(*) FILTER (WHERE has_table_privilege(%s, c.oid, 'INSERT') OR has_table_privilege(%s, c.oid, 'UPDATE')
                           OR has_table_privilege(%s, c.oid, 'DELETE') OR has_table_privilege(%s, c.oid, 'TRUNCATE')) AS reader_write
       FROM pg_class c JOIN pg_namespace n ON n.oid = c.relnamespace
       WHERE n.nspname IN ('cases', 'fraud', 'fraud_gov', 'fraud_dash') AND c.relkind IN ('r', 'v', 'p')
       GROUP BY 1 ORDER BY 1""", (READER,) * 5)
show(cols, rows, f"B.2 Postgres privileges of {READER}")
assert all(r[3] == 0 for r in rows), "reader role has a write privilege"
cols, rows = pg(admin, """SELECT a.attname AS column, has_column_privilege(%s, 'cases.fraud_cases', a.attname, 'SELECT') AS reader_column_select
                          FROM pg_attribute a WHERE a.attrelid = 'cases.fraud_cases'::regclass AND a.attname IN ('notes', 'created_by', 'db_inserted_by')""", (READER,))
show(cols, rows, "B.2 FLAG: pre-existing table-level SELECT of the reader on cases.fraud_cases (rows still hidden by FORCE RLS)")
cols, rows = pg(admin, "SELECT defaclrole::regrole::text AS grantor, defaclnamespace::regnamespace::text AS schema, defaclacl::text AS default_acl FROM pg_default_acl")
show(cols, rows, "Lakebase platform default privileges (source of reader SELECT on new objects)", max_w=120)
assert all(r[1] == "PASS" for r in results), "a deny test failed"
print("PASS: all deny tests")


### B.2 deny tests through UC
| deny test | result | detail |
| --- | --- | --- |
| UC: SELECT count(*) FROM cases.fraud_cases | PASS | (['count(*)'], [[0]]) |
| UC: SELECT notes, db_inserted_by FROM cases.fraud_cases | PASS | (['notes', 'db_inserted_by'], []) |
| UC: SELECT notes, db_inserted_by FROM cases.case_actions | PASS | (['db_inserted_by'], []) |
| UC: SELECT notes FROM fraud_dash.case_summary | PASS | AnalysisException: [UNRESOLVED_COLUMN.WITH_SUGGESTION] A column, variable, or function parameter with name `notes` cannot be resolved. Did you mean one of the f |
| UC: SELECT created_by FROM fraud_dash.case_summary | PASS | AnalysisException: [UNRESOLVED_COLUMN.WITH_SUGGESTION] A column, variable, or function parameter with name `created_by` cannot be resolved. Did you mean one of  |
| UC: SELECT db_inserted_by FROM fraud_dash.case_summary | PASS | AnalysisException: [UNRESOLVED_COLUMN.WITH_SUGGESTION] A column, variable, or function parameter with name `db_inserted_by` cannot

In [0]:
cols, rows = uc_rows(f"SELECT case_id, txn_id, customer_id, status, assigned_queue, recommended_action, risk_score, amount, action_count, created_at FROM {UC}.fraud_dash.case_summary WHERE txn_id = '{TRACED_TXN}'")
show(cols, rows, f"B.3 {TRACED_TXN} through UC fraud_dash.case_summary")
assert rows and rows[0][0].startswith("2319d4b7") and rows[0][4] == "auto_block"
print(f"PASS: {TRACED_TXN} -> case {rows[0][0]}, queue {rows[0][4]}")


### B.3 TXN-0234209 through UC fraud_dash.case_summary
| case_id | txn_id | customer_id | status | assigned_queue | recommended_action | risk_score | amount | action_count | created_at |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| 2319d4b7-bea3-412b-ba6b-8325e1a7a71d | TXN-0234209 | CUST-08669 | open | auto_block | block | 0.9844108757126732 | 323.96 | 1 | 2026-10-21 00:02:21.031588 |
PASS: TXN-0234209 -> case 2319d4b7-bea3-412b-ba6b-8325e1a7a71d, queue auto_block


In [0]:
def pct(xs, p):
    xs = sorted(xs); k = (len(xs) - 1) * p / 100; f = int(k); c = min(f + 1, len(xs) - 1)
    return xs[f] + (xs[c] - xs[f]) * (k - f)

RUNS = 20
LAT = []
print(f"[{now()}] latency start")
for name in DATASETS:
    sql, _, args = dataset_call(name)
    spark.sql(sql, args=args).collect() if args else spark.sql(sql).collect()   # warm-up, not counted
    ts = []
    for _ in range(RUNS):
        t0 = time.time()
        n = len(spark.sql(sql, args=args).collect() if args else spark.sql(sql).collect())
        ts.append((time.time() - t0) * 1000)
    LAT.append([name, DATASETS[name]["display"], n, f"{pct(ts, 50):.0f}", f"{pct(ts, 95):.0f}", f"{min(ts):.0f}", f"{max(ts):.0f}"])
    print(f"  {name}: p50 {pct(ts, 50):.0f} ms")
print(f"[{now()}] latency end")
show(["dataset", "source", "rows", "p50 ms", "p95 ms", "min ms", "max ms"], LAT, f"B.4 dataset latency through UC on serverless ({RUNS} runs each, after 1 warm-up)", max_w=70)

[2026-10-23 23:32:43.501 UTC] latency start
  ds_kpi_cases: p50 2070 ms
  ds_model_vs_baseline: p50 4653 ms
  ds_score_dist: p50 5487 ms
  ds_cases_daily: p50 3255 ms
  ds_actions_daily: p50 2216 ms
  ds_triage: p50 3906 ms
  ds_triage_options: p50 6029 ms
  ds_recent_cases: p50 2030 ms
  ds_calibration: p50 2166 ms
  ds_exposure: p50 5155 ms
[2026-10-23 23:45:55.012 UTC] latency end

### B.4 dataset latency through UC on serverless (20 runs each, after 1 warm-up)
| dataset | source | rows | p50 ms | p95 ms | min ms | max ms |
| --- | --- | --- | --- | --- | --- | --- |
| ds_kpi_cases | Case KPIs (fraud_dash.case_summary) | 1 | 2070 | 3123 | 1061 | 3235 |
| ds_model_vs_baseline | Model vs baseline rule (fraud_dash.baseline_comparison) | 1 | 4653 | 5781 | 2697 | 5841 |
| ds_score_dist | Risk score distribution (fraud_dash.score_distribution) | 31 | 5487 | 8784 | 3852 | 24102 |
| ds_cases_daily | Cases created per day by queue (fraud_dash.case_summary) | 6 | 3255 | 5357 | 1201 | 6240 |
|

In [0]:
print(f"[{now()}] Part D.1 on warehouse {WAREHOUSE_ID}")
md = [f"# Dashboard dataset results ({LOG_DATE})", "",
      f"Dashboard: FE Bar Fraud Ops: Triage & Action (`{DASH_ID}`). Warehouse `{WAREHOUSE_ID}`, Statement Execution API.",
      "SQL is read from the dashboard definition (`serialized_dashboard`). ds_triage runs with its default parameters "
      "(MULTI filters empty, txn_id_f = 'ALL'); only the parameter markers are bound. Generated by validation_notebooks/dashboard_validation.ipynb.", ""]
D1 = {}
for name, d in DATASETS.items():
    sql, params, _ = dataset_call(name)
    cols, rows, ms, sid = wh(sql, params)
    D1[name] = (cols, rows)
    title = f"{name}: {d['display']} ({len(rows)} rows, {ms:.0f} ms, statement {sid})"
    show(cols, rows, title, max_w=70)
    md += [f"## {title}", "", "```sql", d["sql"], "```", "", fmt_table(cols, rows, max_w=120), ""]
md.insert(5, f"Run at {now()}.\n")
out_path = os.path.join(REPO, "logs", f"dashboard_dataset_results_{LOG_DATE}.md")
with open(out_path, "w") as f:
    f.write("\n".join(md) + "\n")
print(f"\nwrote {out_path} ({sum(len(x) for x in md)} chars)")

[2026-10-23 23:46:12.175 UTC] Part D.1 on warehouse 9bbc027402111984

### ds_kpi_cases: Case KPIs (fraud_dash.case_summary) (1 rows, 5118 ms, statement 01f1cf3b-edb4-134e-b0b6-07551a77fadc)
| cases_created_today | open_cases | analyst_review_backlog | visible_cases | as_of_date_utc |
| --- | --- | --- | --- | --- |
| 56 | 8 | 0 | 220 | 2026-10-23 |

### ds_model_vs_baseline: Model vs baseline rule (fraud_dash.baseline_comparison) (1 rows, 4903 ms, statement 01f1cf3b-f0b6-119d-aba5-f5016704ecdc)
| model_fraud_amount_caught | baseline_fraud_amount_caught | model_false_positives | baseline_false_positives | model_flagged | baseline_flagged | baseline_rule_text | total_fraud_amount |
| --- | --- | --- | --- | --- | --- | --- | --- |
| 7010482.03 | 933889.13 | 17630 | 30956 | 24201 | 33723 | is_cross_border = 1 AND is_new_device = 1 | 7986207.92 |

### ds_score_dist: Risk score distribution (fraud_dash.score_distribution) (31 rows, 4225 ms, statement 01f1cf3b-f3a2-1a26-9db2-1e5fe2e26b45)
| 

In [0]:
from fraud_triage_agent import FraudTriageAgent
from mlflow.types.responses import ResponsesAgentRequest
mlflow.set_experiment("/Users/alex.owen@databricks.com/fe_bar_fraud_triage_agent")

KPI_SQL, _, _ = dataset_call("ds_kpi_cases")
T = {}
T["0_start"] = now(); t0_ms = int(time.time() * 1000)

cols, rows, ms, sid_before = wh(KPI_SQL)
T["1_before_query"] = now()
kpi_before = dict(zip(cols, rows[0]))
show(cols, rows, f"BEFORE  ds_kpi_cases  [{T['1_before_query']}]  statement {sid_before}")

cand_sql = (f"SELECT txn_id, customer_id, amount, risk_score, risk_rank, case_status FROM {UC}.fraud_dash.triage_queue "
            "WHERE recommended_action = 'block' AND case_status = 'no case' ORDER BY risk_score DESC, txn_id LIMIT 1")
cols, rows, _, _ = wh(cand_sql)
show(cols, rows, "Candidate: highest-risk block transaction with no case and amount <= 10000 (fraud_dash.triage_queue)")
LIVE_TXN = rows[0][0]
assert float(rows[0][2]) <= 10000
assert not admin.get_cases_for_txn(LIVE_TXN), "candidate already has a case in Postgres"
T["2_candidate"] = now()

# Agent (fraud_triage_agent.ipynb pattern): the LLM decides and calls create_fraud_case,
# which writes through LakebaseCases(pg_role='fraud_triage_agent').
agent = FraudTriageAgent()
t_a = time.time()
resp = agent.predict(ResponsesAgentRequest(input=[{"role": "user", "content": f"Open a case for {LIVE_TXN} and route it."}]))
T["3_agent_done"] = now()
print(f"\nAgent latency {time.time() - t_a:.1f}s, trace {mlflow.get_last_active_trace_id()}")
texts = []
for o in resp.output:
    for c_ in (getattr(o, "content", None) or []):
        if isinstance(c_, dict) and c_.get("text"):
            texts.append(c_["text"])
print("\n".join(texts)[-1500:])
new_cases = admin.get_cases_for_txn(LIVE_TXN)
assert len(new_cases) == 1, new_cases
LIVE_CASE = str(new_cases[0]["case_id"])
print(f"\nPostgres write: case {LIVE_CASE}, created_by={new_cases[0]['created_by']}, db_inserted_by={new_cases[0]['db_inserted_by']}, created_at={new_cases[0]['created_at']}")

# Same dataset SQL again, immediately, no sync or job in between
cols, rows, ms, sid_after = wh(KPI_SQL)
T["4_after_query"] = now()
kpi_after = dict(zip(cols, rows[0]))
show(cols, rows, f"AFTER   ds_kpi_cases  [{T['4_after_query']}]  statement {sid_after}")

cols, rows, _, _ = wh(f"SELECT case_id, txn_id, customer_id, status, assigned_queue, recommended_action, risk_score, amount, action_count, created_at FROM {UC}.fraud_dash.case_summary WHERE txn_id = '{LIVE_TXN}'")
T["5_case_row"] = now()
show(cols, rows, f"New case row from fraud_dash.case_summary via warehouse [{T['5_case_row']}]")
assert rows and rows[0][0] == LIVE_CASE

show(["metric", "before", "after", "delta"],
     [[k, kpi_before[k], kpi_after[k], int(kpi_after[k]) - int(kpi_before[k])] for k in ["cases_created_today", "open_cases", "visible_cases"]],
     "Before vs after (same ds_kpi_cases SQL the dashboard counters run)")

# Nothing ran in between: job runs and pipeline updates since the start of this cell
t1_ms = int(time.time() * 1000)
job_runs = [r for r in w.jobs.list_runs(job_id=JOB_ID, start_time_from=t0_ms, start_time_to=t1_ms)]
upd = [u for u in (w.pipelines.list_updates(pipeline_id=SYNC_PIPELINE, max_results=5).updates or []) if (u.creation_time or 0) >= t0_ms]
last = (w.pipelines.list_updates(pipeline_id=SYNC_PIPELINE, max_results=1).updates or [None])[0]
show(["check", "value"], [[f"job {JOB_ID} runs started in window", len(job_runs)],
                         [f"sync pipeline {SYNC_PIPELINE} updates created in window", len(upd)],
                         ["latest sync pipeline update", f"{last.update_id} {last.state} created {dt.datetime.fromtimestamp(last.creation_time/1000, dt.timezone.utc):%Y-%m-%d %H:%M:%S} UTC" if last else "none"]],
     "No sync or job run between before and after")
assert not job_runs and not upd
show(["step", "timestamp"], sorted(T.items()), "Timestamps")
assert int(kpi_after["cases_created_today"]) == int(kpi_before["cases_created_today"]) + 1
assert int(kpi_after["open_cases"]) == int(kpi_before["open_cases"]) + 1
print(f"\nPASS: live read. case {LIVE_CASE} for {LIVE_TXN} is visible to the dashboard's dataset queries with no sync or job.")

/local_disk0/.ephemeral_nfs/envs/pythonEnv-d377472e-6266-488e-a2f9-3bd4950629f7/lib/python3.12/site-packages/langgraph/checkpoint/base/__init__.py:18: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer
2026/10/23 23:46:53 WARNING mlflow.utils.autologging_utils: MLflow langchain autologging is known to be compatible with 1.0.0 <= langchain, but the installed version is 0.3.30. If you encounter errors during autologging, try upgrading / downgrading langchain to a compatible version, or try upgrading MLflow.
If you are using MLflow Tracing, you can migrate your traces to Unity Catalog for unlimited storage, fine-grained access controls, and queryability from notebooks, SQL, and dashboards. Learn more: https://docs.databricks.com/aws/en/mlflow3/genai/tracing


### BEFORE  ds_kpi_cases  [2026-10-23 23:46:55.954 UTC]  statement 01f1cf3c-067f-1949-8dd1-e42fa892303b
| cases_created_today | open_cases | analyst_review_backlog | visible_cases | as_of_date_utc |
| --- | --- | --- | --- | --- |
| 56 | 8 | 0 | 220 | 2026-10-23 |

### Candidate: highest-risk block transaction with no case and amount <= 10000 (fraud_dash.triage_queue)
| txn_id | customer_id | amount | risk_score | risk_rank | case_status |
| --- | --- | --- | --- | --- | --- |
| TXN-0112255 | CUST-09364 | 169.69 | 0.9924971554250681 | 8 | no case |


2026/10/23 23:46:59 WARNING mlflow.tracking.context.registry: Encountered unexpected error during resolving tags: getContext().extraContext() is not a CommandContext accessor, it is not available or is misspelled.
/local_disk0/.ephemeral_nfs/envs/pythonEnv-d377472e-6266-488e-a2f9-3bd4950629f7/lib/python3.12/site-packages/pydantic/main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `str` - serialized value may not be as expected [field_name='content', input_value=[{'type': 'reasoning', 's...ZvQ48K9tcvmRijPGAE='}]}], input_type=list])
  return self.__pydantic_serializer__.to_python(
/local_disk0/.ephemeral_nfs/envs/pythonEnv-d377472e-6266-488e-a2f9-3bd4950629f7/lib/python3.12/site-packages/pydantic/main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `str` - serialized value may not be as expected [field_name='content', input_value=[{'type': 'reasoning', 's...CC+TXkJJj7TBXIYAQ=='}]}], i


Agent latency 18.6s, trace tr-9a61d00c74d1651ee0e115927e4c4378
d:** `explain_top_signals` returned an error ("Unknown format code 'f' for object of type 'str'") on two tries. I won't guess at the drivers. I noted the missing explanation in the case notes. The tool will need fixing before you can get SHAP detail on this transaction.

**Transaction:** $169.69 on 2026-09-23 for CUST-09364. The risk score is **0.9925**.

**Recommended action:** Block. This is the model's action, and I didn't override it.

**Confidence:** Medium. The observed precision is 0.50 in the block band (n=288). Roughly half of transactions scored in this band turned out to be fraud, so the high score is weaker evidence than it looks.

**Routed queue:** auto_block.

**Customer context:**
- The customer is in the high risk segment, with 52 months of tenure (the tool gave no unit) and a US-East home region.
- 7 of their 11 prior cases were confirmed fraud.
- In the last 30 days they used 5 devices in 5 countries, and

Trace(trace_id='tr-9a61d00c74d1651ee0e115927e4c4378')

In [0]:
print(f"[{now()}] D.3 end-to-end trace for {TRACED_TXN}")
cols, rows, _, _ = wh(f"SELECT * FROM {UC}.fraud_gov.txn_risk_current WHERE txn_id = '{TRACED_TXN}'")
show(cols, rows, "fraud_gov.txn_risk_current (warehouse)", max_w=200)
cols, rows, _, _ = wh(f"SELECT * FROM {UC}.fraud_dash.case_summary WHERE txn_id = '{TRACED_TXN}'")
show(cols, rows, "fraud_dash.case_summary (warehouse)")
sql, params, _ = dataset_call("ds_triage", txn_id=TRACED_TXN)
cols, rows, _, _ = wh(sql, params)
show(cols, rows, f"Triage queue dataset ds_triage with txn_id filter = {TRACED_TXN} (warehouse)", max_w=200)
assert len(rows) == 1 and rows[0][0] == TRACED_TXN

# Journey: raw -> silver -> gold -> score -> Lakebase -> agent -> case -> dashboard
CAT = "workshop_guy_catalog.fe_bar_a0"
hops = [
    ("raw/bronze", f"SELECT count(*) FROM {CAT}.fact_transactions_bronze WHERE txn_id = '{TRACED_TXN}'"),
    ("silver", f"SELECT count(*) FROM {CAT}.fact_transactions_silver WHERE txn_id = '{TRACED_TXN}'"),
    ("gold features", f"SELECT count(*) FROM {CAT}.txn_features_gold WHERE txn_id = '{TRACED_TXN}'"),
    ("score (Delta)", f"SELECT concat(round(risk_score, 5), ' ', recommended_action) FROM {CAT}.scored_transactions WHERE txn_id = '{TRACED_TXN}'"),
    ("Lakebase synced", f"SELECT concat(round(risk_score, 5), ' ', recommended_action) FROM {UC}.fraud.txn_risk_current WHERE txn_id = '{TRACED_TXN}'"),
    ("Lakebase governed", f"SELECT concat(round(risk_score, 5), ' ', recommended_action) FROM {UC}.fraud_gov.txn_risk_current WHERE txn_id = '{TRACED_TXN}'"),
    ("dashboard case_summary", f"SELECT concat(case_id, ' ', assigned_queue, ' ', status) FROM {UC}.fraud_dash.case_summary WHERE txn_id = '{TRACED_TXN}'"),
]
journey = []
for hop, q_ in hops:
    try: journey.append([hop, uc_rows(q_)[1][0][0]])
    except Exception as e: journey.append([hop, f"ERR {type(e).__name__}: {str(e).splitlines()[0][:100]}"])
c = admin.get_cases_for_txn(TRACED_TXN)[0]
journey.insert(6, ["agent -> case (Postgres)", f"{c['case_id']} created_by={c['created_by']} db_inserted_by={c['db_inserted_by']}"])
show(["hop", "evidence for " + TRACED_TXN], journey, "Journey raw -> silver -> gold -> score -> Lakebase -> agent -> case -> dashboard", max_w=120)

[2026-10-23 23:47:27.408 UTC] D.3 end-to-end trace for TXN-0234209

### fraud_gov.txn_risk_current (warehouse)
| txn_id | customer_id | risk_score | recommended_action | top_shap_signals | model_version | scored_at |
| --- | --- | --- | --- | --- | --- | --- |
| TXN-0234209 | CUST-08669 | 0.9844108757126732 | block | [{"feature": "is_new_device", "value": 1.0, "shap": 2.5724}, {"feature": "geo_distance_from_home", "value": 450.63, "shap": 1.8741}, {"feature": "is_online", "value": 1.0, "shap": 1.2001}] | 1 | 2026-10-21T02:36:46.583Z |

### fraud_dash.case_summary (warehouse)
| case_id | txn_id | customer_id | status | assigned_queue | recommended_action | created_at | updated_at | risk_score | amount | action_count | last_action_at |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| 2319d4b7-bea3-412b-ba6b-8325e1a7a71d | TXN-0234209 | CUST-08669 | open | auto_block | block | 2026-10-21T00:02:21.031Z | 2026-10-21T00:02:21.031Z | 0.9844108757126732 | 323.96 | 1 

In [0]:
summary = {
    "run_at": now(), "dashboard_id": DASH_ID, "warehouse_id": WAREHOUSE_ID, "uc_reader_role": READER,
    "uc_cases_raw": UC_CASES_BEFORE, "dash_cases": DASH_CASES, "lead_cases_at_B1": LEAD_CASES,
    "thresholds": {"block": BLOCK_T, "review": REVIEW_T, "run": MLFLOW_RUN},
    "deny_tests": {r[0]: r[1] for r in results},
    "latency_ms": {r[0]: {"p50": r[3], "p95": r[4]} for r in LAT},
    "live_proof": {"txn_id": LIVE_TXN, "case_id": LIVE_CASE, "before": kpi_before, "after": kpi_after, "timestamps": T},
}
print(json.dumps(summary, indent=2, default=str))
admin.close(); lead.close()

{
  "run_at": "2026-10-23 23:47:55.476 UTC",
  "dashboard_id": "01f1ce7f835a1e81b05ebb664bf0994c",
  "warehouse_id": "9bbc027402111984",
  "uc_reader_role": "databricks_reader_17063",
  "uc_cases_raw": 0,
  "dash_cases": 220,
  "lead_cases_at_B1": 226,
  "thresholds": {
    "block": 0.9502674433961342,
    "review": 0.8714158290727556,
    "run": "d89beaca8f81447780da60ed2ea7c811"
  },
  "deny_tests": {
    "UC: SELECT count(*) FROM cases.fraud_cases": "PASS",
    "UC: SELECT notes, db_inserted_by FROM cases.fraud_cases": "PASS",
    "UC: SELECT notes, db_inserted_by FROM cases.case_actions": "PASS",
    "UC: SELECT notes FROM fraud_dash.case_summary": "PASS",
    "UC: SELECT created_by FROM fraud_dash.case_summary": "PASS",
    "UC: SELECT db_inserted_by FROM fraud_dash.case_summary": "PASS",
    "UC: INSERT INTO fraud_dash.model_thresholds": "PASS",
    "UC: UPDATE cases.fraud_cases SET notes = notes": "PASS",
    "UC: DELETE FROM fraud_dash.case_summary WHERE false": "PASS"
  },
  "